# Day 6: Ranking, Walked Through

Retrieval finds candidates. Ranking decides the order they're shown in -- and
order is often the difference between a good answer and a wrong one. This notebook
walks through scoring signals, combining them, measuring quality, and proving that
ranking matters as much as retrieval.

Run each cell in order. Plain Python throughout, self-contained.

## Cell 1: What is ranking and why it matters

Google technically matches millions of pages for a search -- but you only ever
look at the first few results. Ranking is what decides which handful of results
actually get seen. In RAG, an LLM only gets the top few retrieved chunks, so if
ranking puts the truly relevant chunk at position 8, the LLM never sees it at all
-- functionally identical to retrieval having missed it completely.

## Cell 2: Different scoring strategies

Similarity is one signal. Recency and quality are others. Let's see each on its own.

In [ ]:
import math

def recency_score(updated_days_ago, half_life_days=30):
    return math.exp(-updated_days_ago * math.log(2) / half_life_days)

def quality_score(rating, max_rating=5.0):
    return max(0.0, min(1.0, rating / max_rating))

def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    ma = math.sqrt(sum(x * x for x in a))
    mb = math.sqrt(sum(y * y for y in b))
    return dot / (ma * mb) if ma and mb else 0.0

print("Recency score by age:")
for days in [0, 15, 30, 90]:
    print(f"  {days:>3} days ago: {recency_score(days):.3f}")

print("\nQuality score by rating:")
for rating in [5.0, 3.5, 2.0]:
    print(f"  {rating}/5: {quality_score(rating):.3f}")

## Cell 3: Combine multiple signals

No single signal tells the whole story. Let's build a small set of candidates where
similarity, recency, and quality each favor a DIFFERENT document, then combine them
with weights.

In [ ]:
candidates = [
    {"title": "Doc A (best match, old, mediocre)", "similarity": 0.95, "updated_days_ago": 300, "rating": 3.0},
    {"title": "Doc B (weak match, very new, ok)", "similarity": 0.55, "updated_days_ago": 1, "rating": 3.5},
    {"title": "Doc C (ok match, medium age, best)", "similarity": 0.70, "updated_days_ago": 45, "rating": 4.8},
]

for c in candidates:
    c["recency_score"] = recency_score(c["updated_days_ago"])
    c["quality_score"] = quality_score(c["rating"])

def weighted_rank(candidates, weights):
    ranked = [{**c, "final_score": c["similarity"] * weights["similarity"] + c["recency_score"] * weights["recency"] + c["quality_score"] * weights["quality"]} for c in candidates]
    return sorted(ranked, key=lambda c: c["final_score"], reverse=True)

for name, weights in [("Similarity-only", {"similarity": 1.0, "recency": 0.0, "quality": 0.0}),
                       ("Balanced (50/30/20)", {"similarity": 0.5, "recency": 0.3, "quality": 0.2})]:
    print(f"--- {name} ---")
    for r in weighted_rank(candidates, weights):
        print(f"  {r['title']:38s} final={r['final_score']:.3f}")
    print()

Notice the winner changes depending on the weights -- pure similarity always picks
Doc A, even though it's 300 days stale. A balanced weighting picks something else.

## Cell 4: Measure quality with metrics

NDCG, MRR, and MAP turn "this ranking looks better" into an actual number.

In [ ]:
def dcg_at_k(ranked_ids, relevance_scores, k):
    return sum(relevance_scores.get(doc_id, 0) / math.log2(i + 2) for i, doc_id in enumerate(ranked_ids[:k]))

def ndcg_at_k(ranked_ids, relevance_scores, k):
    actual = dcg_at_k(ranked_ids, relevance_scores, k)
    ideal_order = sorted(relevance_scores, key=lambda d: relevance_scores[d], reverse=True)
    ideal = dcg_at_k(ideal_order, relevance_scores, k)
    return actual / ideal if ideal else 1.0

def reciprocal_rank(ranked_ids, relevant_ids):
    for i, doc_id in enumerate(ranked_ids):
        if doc_id in relevant_ids:
            return 1.0 / (i + 1)
    return 0.0

def average_precision(ranked_ids, relevant_ids):
    if not relevant_ids:
        return 1.0
    hits, precisions = 0, []
    for i, doc_id in enumerate(ranked_ids):
        if doc_id in relevant_ids:
            hits += 1
            precisions.append(hits / (i + 1))
    return sum(precisions) / len(relevant_ids) if precisions else 0.0

relevance_scores = {2: 2, 5: 1}
good_ranking = [2, 5, 3]
bad_ranking = [3, 3, 2]  # 5 is missing entirely from this ranking

print(f"Good ranking -- NDCG@3: {ndcg_at_k(good_ranking, relevance_scores, 3):.2f}, MRR: {reciprocal_rank(good_ranking, {2,5}):.2f}, MAP: {average_precision(good_ranking, {2,5}):.2f}")
print(f"Bad ranking  -- NDCG@3: {ndcg_at_k(bad_ranking, relevance_scores, 3):.2f}, MRR: {reciprocal_rank(bad_ranking, {2,5}):.2f}, MAP: {average_precision(bad_ranking, {2,5}):.2f}")

## Cell 5: Advanced ranking techniques

Diversity ranking avoids showing several near-duplicate results in a row.

In [ ]:
def diversity_rerank(ranked_results, similarity_between_fn, diversity_penalty=0.5):
    remaining = list(ranked_results)
    selected = []
    while remaining:
        best_candidate, best_score = None, -math.inf
        for candidate in remaining:
            penalty = max((similarity_between_fn(candidate, s) for s in selected), default=0.0)
            adjusted = candidate["score"] - penalty * diversity_penalty
            if adjusted > best_score:
                best_score, best_candidate = adjusted, candidate
        selected.append(best_candidate)
        remaining.remove(best_candidate)
    return selected

results = [
    {"title": "RAG intro (v1)", "score": 0.95, "topic": "rag"},
    {"title": "RAG intro (v2, near-dup)", "score": 0.93, "topic": "rag"},
    {"title": "Vector DB overview", "score": 0.80, "topic": "vectordb"},
]
same_topic = lambda a, b: 1.0 if a["topic"] == b["topic"] else 0.0

print("Before diversity re-rank:", [r["title"] for r in sorted(results, key=lambda r: r["score"], reverse=True)])
print("After diversity re-rank: ", [r["title"] for r in diversity_rerank(results, same_topic)])

## Cell 6: Compare ranking quality with and without good ranking

Same retrieved pool, different order -- let's measure the real difference.

In [ ]:
import random

# Same underlying pool of doc IDs (unique documents, not raw chunks --
# ranking metrics assume each relevant ID can only be "found" once).
pool = [2, 5, 3, 6, 1]
relevant_ids = {2, 5}
relevance_scores = {2: 2, 5: 1}

random.seed(3)
no_ranking = list(pool)
random.shuffle(no_ranking)

with_ranking = [2, 5, 3, 6, 1]  # what a good ranker should produce: relevant docs first

print("No ranking (random):  ", no_ranking)
print("With ranking (good):  ", with_ranking)
print()

for name, ranked in [("No ranking", no_ranking), ("With ranking", with_ranking)]:
    ndcg = ndcg_at_k(ranked, relevance_scores, 3)
    mrr = reciprocal_rank(ranked, relevant_ids)
    mapv = average_precision(ranked, relevant_ids)
    print(f"{name:14s} -- NDCG@3: {ndcg:.3f}, MRR: {mrr:.3f}, MAP: {mapv:.3f}")

## Cell 7: Complete integration example

Let's run the full pipeline: chunk, embed, store, retrieve, and rank -- all six days
in one flow.

In [ ]:
CONCEPTS = {
    "programming":   ["python", "programming", "language", "syntax", "code"],
    "data_types":    ["data", "types", "strings", "integers", "floats", "lists"],
    "retrieval":     ["rag", "retrieval", "retrieves", "generation", "lookup", "find"],
    "freshness":     ["outdated", "fresh", "matters", "old", "problem"],
    "search_tech":   ["vector", "database", "embeddings", "similarity", "search", "meaning"],
}

def text_to_embedding(text):
    words = set(w.strip("?.,!").lower() for w in text.split())
    return [float(sum(1 for w in words if w in cw)) for cw in CONCEPTS.values()]

def chunk_by_paragraphs(text):
    return [p.strip() for p in text.split("\n\n") if p.strip()]

documents = [
    {"id": 2, "title": "What is RAG", "updated_days_ago": 5, "rating": 4.5,
     "text": "RAG stands for Retrieval-Augmented Generation. It retrieves relevant documents before generating an answer.\n\nThis approach retrieves information first, then generates a grounded response."},
    {"id": 3, "title": "What is a Vector Database", "updated_days_ago": 60, "rating": 4.8,
     "text": "A vector database stores data as numerical vectors called embeddings. It allows fast similarity search."},
    {"id": 5, "title": "Why RAG Matters", "updated_days_ago": 2, "rating": 3.5,
     "text": "RAG matters because language models can be outdated. Retrieval helps fix that problem by finding fresh information."},
]

vector_db = []  # (embedding, metadata)
for doc in documents:
    for i, chunk in enumerate(chunk_by_paragraphs(doc["text"])):
        emb = text_to_embedding(doc["title"] + " " + chunk)
        vector_db.append((emb, {"doc_id": doc["id"], "title": doc["title"], "chunk_index": i,
                                 "updated_days_ago": doc["updated_days_ago"], "rating": doc["rating"]}))

def retrieve(query, top_k=5):
    q_emb = text_to_embedding(query)
    scored = [(meta, cosine_similarity(q_emb, emb)) for emb, meta in vector_db]
    scored.sort(key=lambda pair: pair[1], reverse=True)
    return [{"metadata": m, "similarity": s} for m, s in scored[:top_k] if s > 0]

def rank(results, weights={"similarity": 0.5, "recency": 0.3, "quality": 0.2}):
    scored = []
    for r in results:
        rec = recency_score(r["metadata"]["updated_days_ago"])
        qual = quality_score(r["metadata"]["rating"])
        final = r["similarity"] * weights["similarity"] + rec * weights["recency"] + qual * weights["quality"]
        scored.append({**r, "final_score": final})
    return sorted(scored, key=lambda r: r["final_score"], reverse=True)

# "matters" and "problem" pull in "Why RAG Matters" too, alongside "What is
# RAG" -- giving retrieval more than one document to actually choose between.
query = "Why does RAG matter and fix problems?"
retrieved = retrieve(query)
ranked = rank(retrieved)

print(f"Query: {query}\n")
print("Retrieved (by similarity):")
for r in retrieved:
    print(f"  {r['metadata']['title']:26s} chunk {r['metadata']['chunk_index']} (similarity {r['similarity']:.3f})")
print("\nRanked (by combined score):")
for r in ranked:
    print(f"  {r['metadata']['title']:26s} chunk {r['metadata']['chunk_index']} (final {r['final_score']:.3f})")

## Cell 8: Key takeaways and best practices

- Ranking decides the final order retrieval's candidates are shown in -- and since
  users (and LLMs) only see the top few results, ranking quality is just as
  important as retrieval quality.
- No single signal (similarity, recency, quality) tells the whole story on its own.
  Combining them, with tunable weights, produces better results than any one alone.
- Ranking quality is measurable: NDCG rewards putting the MOST relevant results at
  the top, MRR cares about the first relevant hit, MAP rewards finding ALL relevant
  results near the top.
- Advanced techniques (diversity, context-awareness, temporal weighting,
  domain-specific rules) layer real-world judgment on top of raw similarity.
- Watch for double-counting: if retrieval returns multiple chunks from the same
  document, collapse to unique documents before computing NDCG/MRR/MAP, or scores
  can be inflated past their intended range.
- A weight that helps one kind of query (e.g. a strong recency boost for
  time-sensitive questions) can actively hurt a different kind of query (a timeless
  factual question) -- weights need evaluation, not just intuition.
- Retrieval finds a good POOL. Ranking decides the best final ORDER. Generation
  (the next stage of RAG) can only work with whatever ranking hands it.